### Data Generation and Initital Setup

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import *
from pyspark.sql.functions import col
import random
from datetime import datetime, timedelta

# 1. Define schema with strict types
schema = StructType([
    StructField("InvoiceNo", StringType()),
    StructField("StockCode", StringType()),
    StructField("Description", StringType()),
    StructField("Quantity", IntegerType()),
    StructField("InvoiceDate", TimestampType()),
    StructField("UnitPrice", DoubleType()),
    StructField("CustomerID", StringType()),
    StructField("Country", StringType())
])

# 2. Product catalog with prices as floats
products = [
    ("85123A", "WHITE HANGING HEART T-LIGHT HOLDER", 2.55),
    ("71053", "WHITE METAL LANTERN", 3.39),
    ("84406B", "CREAM CUPID HEARTS COAT HANGER", 2.75),
    ("84029G", "KNITTED UNION FLAG HOT WATER BOTTLE", 3.75),
    ("84029E", "RED WOOLLY HOTTIE WHITE HEART", 3.75)
]

# 3. Generate clean data
def generate_record(record_id):
    product = random.choice(products)
    return Row(
        InvoiceNo=f"C{record_id:05d}",  # Proper string formatting
        StockCode=product[0],
        Description=product[1],
        Quantity=random.randint(1, 10),
        InvoiceDate=datetime(2023, 1, 1) + timedelta(days=random.randint(0, 365)),
        UnitPrice=float(round(product[2] * random.uniform(0.9, 1.1), 2)),  # Explicit float
        CustomerID=f"C{random.randint(1000, 9999)}",
        Country=random.choice(["UK", "France", "Germany", "USA", "Australia", "Japan"])
    )

# 4. Create DataFrame
df = spark.createDataFrame([generate_record(i) for i in range(1, 101)], schema)

# 5. Verify types
print("Schema verification:")
df.printSchema()

# 6. Show sample
display(df.limit(5))

Schema verification:
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Country: string (nullable = true)



InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
C00001,71053,WHITE METAL LANTERN,7,2023-04-15T00:00:00.000+0000,3.18,C4231,Australia
C00002,84406B,CREAM CUPID HEARTS COAT HANGER,7,2023-07-08T00:00:00.000+0000,2.56,C4171,USA
C00003,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2023-08-01T00:00:00.000+0000,2.62,C4369,UK
C00004,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2,2023-01-04T00:00:00.000+0000,2.78,C8364,USA
C00005,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,10,2023-09-29T00:00:00.000+0000,3.4,C3619,USA


In [0]:
from pyspark.sql.functions import sum, round

# SAFE analysis with explicit type casting
analysis_df = df.groupBy("Country").agg(
    sum("Quantity").alias("TotalItemsSold"),
    round(sum(col("Quantity").cast("double") * col("UnitPrice")), 2).alias("TotalRevenue")
).orderBy("TotalRevenue", ascending=False)

display(analysis_df)

Country,TotalItemsSold,TotalRevenue
USA,144,461.57
Japan,94,311.16
France,92,298.97
Germany,75,251.97
UK,73,235.51
Australia,70,225.61


### Country-Level Sales Analysis

In [0]:
from pyspark.sql.functions import sum, desc

country_sales = df.groupBy("Country") \
    .agg(sum("Quantity").alias("TotalItemsSold"),
         sum(df.Quantity * df.UnitPrice).alias("TotalRevenue")) \
    .orderBy(desc("TotalRevenue"))

display(country_sales)

Country,TotalItemsSold,TotalRevenue
USA,144,461.57
Japan,94,311.15999999999997
France,92,298.96999999999997
Germany,75,251.97000000000003
UK,73,235.51000000000002
Australia,70,225.61


### Product Performance Analysis

In [0]:
top_products = df.groupBy("Description") \
    .agg(sum("Quantity").alias("UnitsSold"),
         sum(df.Quantity * df.UnitPrice).alias("Revenue")) \
    .orderBy(desc("Revenue")) \
    .limit(10)

display(top_products)

Description,UnitsSold,Revenue
RED WOOLLY HOTTIE WHITE HEART,111,406.54
KNITTED UNION FLAG HOT WATER BOTTLE,109,406.5
WHITE METAL LANTERN,116,402.74
CREAM CUPID HEARTS COAT HANGER,139,385.34000000000003
WHITE HANGING HEART T-LIGHT HOLDER,73,183.67000000000002


### Monthly Performance Analysis

In [0]:
from pyspark.sql.functions import month, year

monthly_trend = df.groupBy(year("InvoiceDate").alias("Year"),
                          month("InvoiceDate").alias("Month")) \
    .agg(sum(df.Quantity * df.UnitPrice).alias("MonthlyRevenue")) \
    .orderBy("Year", "Month")

display(monthly_trend)

Year,Month,MonthlyRevenue
2023,1,122.36000000000001
2023,2,297.97
2023,3,68.0
2023,4,131.55
2023,5,103.97
2023,6,173.84
2023,7,135.96
2023,8,162.65999999999997
2023,9,91.68
2023,10,238.26000000000002


### Daily Sales Dashboard

In [0]:
from pyspark.sql.functions import date_format

# 1. Prepare time-series data
daily_sales = df.groupBy(date_format("InvoiceDate", "yyyy-MM-dd").alias("Date")) \
               .agg(sum("Quantity").alias("TotalItems"), 
                    sum(col("Quantity")*col("UnitPrice")).alias("Revenue")) \
               .orderBy("Date")

# 2. Display as an interactive dashboard
display(daily_sales)

Date,TotalItems,Revenue
2023-01-02,10,29.14
2023-01-03,3,11.73
2023-01-04,8,22.26
2023-01-07,1,3.36
2023-01-15,7,21.349999999999998
2023-01-19,9,30.87
2023-01-31,1,3.65
2023-02-03,9,33.03
2023-02-04,4,10.36
2023-02-05,3,11.28


Databricks visualization. Run in Databricks to view.

Sales trend visualisation 

In [0]:
from pyspark.sql.functions import date_format

# Prepare daily sales data
daily_sales = df.groupBy(date_format("InvoiceDate", "yyyy-MM-dd").alias("Date")) \
               .agg(sum("Quantity").alias("ItemsSold"), 
                    sum(col("Quantity")*col("UnitPrice")).alias("Revenue")) \
               .orderBy("Date")

# Display with interactive line chart
display(daily_sales)

Date,ItemsSold,Revenue
2023-01-02,10,29.14
2023-01-03,3,11.73
2023-01-04,8,22.26
2023-01-07,1,3.36
2023-01-15,7,21.349999999999998
2023-01-19,9,30.87
2023-01-31,1,3.65
2023-02-03,9,33.03
2023-02-04,4,10.36
2023-02-05,3,11.28


Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import count, sum, max, datediff, current_date, col, when

# Ensure your DataFrame has the correct column names
print("Available columns:", df.columns)

rfm = df.groupBy("CustomerID") \
       .agg(
           count("InvoiceNo").alias("Frequency"),
           sum(col("Quantity") * col("UnitPrice")).alias("MonetaryValue"),
           datediff(current_date(), max("InvoiceDate")).alias("Recency")
       ) \
       .filter(col("CustomerID").isNotNull())

# Customer segmentation logic
rfm_segments = rfm.withColumn("Segment",
    when(col("Recency") > 90, "At Risk")
    .when((col("Recency") <= 90) & (col("Frequency") < 5), "Needs Attention")
    .when((col("Recency") <= 60) & (col("MonetaryValue") > 500), "Champions")
    .otherwise("Loyal")
)

display(rfm_segments.orderBy("MonetaryValue", ascending=False))

Available columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


CustomerID,Frequency,MonetaryValue,Recency,Segment
C2035,1,39.1,702,At Risk
C8747,1,36.9,800,At Risk
C8874,1,36.0,764,At Risk
C8827,1,36.0,520,At Risk
C4962,1,34.7,696,At Risk
C2505,1,34.5,655,At Risk
C9252,1,34.4,626,At Risk
C3619,1,34.0,582,At Risk
C9487,1,33.8,636,At Risk
C3087,1,33.120000000000005,500,At Risk


Databricks visualization. Run in Databricks to view.

Product Analysis

In [0]:
from pyspark.sql.functions import col, sum, countDistinct, when

# 1. Top Products Analysis
product_performance = df.groupBy("StockCode", "Description").agg(
    sum("Quantity").alias("TotalUnitsSold"),
    sum(col("Quantity")*col("UnitPrice")).alias("TotalRevenue"),
    countDistinct("InvoiceNo").alias("PurchaseOccasions"),
    (sum(col("Quantity")*col("UnitPrice")) / countDistinct("InvoiceNo")).alias("AvgOrderValue")
).orderBy(col("TotalRevenue").desc())

# 2. ABC Analysis (Pareto)
product_performance = product_performance.withColumn(
    "RevenuePercentage",
    col("TotalRevenue") / product_performance.agg(sum("TotalRevenue")).first()[0] * 100
)

product_performance = product_performance.withColumn(
    "ProductCategory",
    when(col("RevenuePercentage") > 5, "A (Top 20%)")
    .when(col("RevenuePercentage") > 2, "B (Next 30%)")
    .otherwise("C (Bottom 50%)")
)

# 3. Display as interactive dashboard
display(product_performance)

StockCode,Description,TotalUnitsSold,TotalRevenue,PurchaseOccasions,AvgOrderValue,RevenuePercentage,ProductCategory
84029E,RED WOOLLY HOTTIE WHITE HEART,111,406.53999999999996,20,20.326999999999998,22.778029908280526,A (Top 20%)
84029G,KNITTED UNION FLAG HOT WATER BOTTLE,109,406.5,20,20.325,22.775788748256097,A (Top 20%)
71053,WHITE METAL LANTERN,116,402.74,19,21.19684210526316,22.565119705959805,A (Top 20%)
84406B,CREAM CUPID HEARTS COAT HANGER,139,385.3399999999999,24,16.05583333333333,21.59021509533334,A (Top 20%)
85123A,WHITE HANGING HEART T-LIGHT HOLDER,73,183.67000000000002,17,10.804117647058824,10.290846542170229,A (Top 20%)
